# _lib/type_helpers — Silver type-fix hulpfuncties

Pure hulpfuncties voor de Bronze→Silver type-casts.
Herbruikbaar voor `order_detail` en toekomstige Silver-tabellen.

**Importeer via `%run ./_lib/type_helpers` vanuit het Silver DLT-pipeline-notebook.**

| Bronze-kolom | Bronze-type | Silver-type |
|---|---|---|
| `SERVED_TS` | `StringType` | `TimestampType` |
| `ORDER_TAX_AMOUNT` | `StringType` | `DecimalType(38, 4)` |
| `ORDER_DISCOUNT_AMOUNT` | `StringType` | `DecimalType(38, 4)` |
| `ORDER_ITEM_DISCOUNT_AMOUNT` | `StringType` | `DecimalType(38, 4)` |
| `LOCATION_ID` | `DoubleType` | `DecimalType(38, 0)` |
| `DISCOUNT_ID` | `StringType` | `DecimalType(38, 0)` nullable |
| `SHIFT_START_TIME` | `IntegerType` (millis) | `StringType` `'HH:mm:ss'` |
| `SHIFT_END_TIME` | `IntegerType` (millis) | `StringType` `'HH:mm:ss'` |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Column


def millis_to_hhmmss(col: Column) -> Column:
    """Converteert een IntegerType-kolom van milliseconden-sinds-middernacht naar
    een 'HH:mm:ss' StringType-kolom.

    Bronze slaat SHIFT_START_TIME en SHIFT_END_TIME op als ruwe INT32
    TIME(MILLIS)-waarden.  Silver wil leesbare 'HH:mm:ss'-strings.

    Example: 36000000 ms  →  '10:00:00'
    """
    total_seconds = (col / 1000).cast("long")
    hh = F.lpad((total_seconds / 3600).cast("int").cast("string"), 2, "0")
    mm = F.lpad(((total_seconds % 3600) / 60).cast("int").cast("string"), 2, "0")
    ss = F.lpad((total_seconds % 60).cast("int").cast("string"), 2, "0")
    return F.concat(hh, F.lit(":"), mm, F.lit(":"), ss)


def cast_served_ts(col: Column) -> Column:
    """Parseert SERVED_TS (StringType 'yyyy-MM-dd HH:mm:ss') naar TimestampType."""
    return F.to_timestamp(col, "yyyy-MM-dd HH:mm:ss")


def cast_decimal_38_4(col: Column) -> Column:
    """Cast een StringType-bedragkolom naar DecimalType(38, 4)."""
    return col.cast("decimal(38, 4)")


def cast_decimal_38_0(col: Column) -> Column:
    """Cast een DoubleType- of StringType-ID-kolom naar DecimalType(38, 0).

    Gebruikt voor LOCATION_ID (Double→Decimal) en DISCOUNT_ID (String→Decimal nullable).
    Geeft NULL terug voor NULL-invoer (nullable-semantiek blijft behouden).
    """
    return col.cast("decimal(38, 0)")


print("type_helpers geladen: millis_to_hhmmss, cast_served_ts, cast_decimal_38_4, cast_decimal_38_0")